# 🎧 NexTune — Bluetooth Headphones Price Prediction
### Model Training Notebook

**Problem Statement:** Recommend a suitable price for a new wireless Bluetooth headphone in the Indian market based on its specifications and market demand (Amazon India).

**Pipeline:**
1. Load & clean the scraped dataset
2. Feature engineering
3. Train multiple regression models
4. Evaluate & select the best model
5. Feature importance analysis
6. Save the trained model
7. Predict price for a new headphone

## ⚙️ Setup

In [ ]:
# Install dependencies (Google Colab)
!pip install -q pandas numpy scikit-learn matplotlib seaborn joblib

In [ ]:
# Mount Google Drive (to access dataset and save model)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('✅ All libraries loaded')

## 1. Load Dataset

In [ ]:
# Option A: Load from GitHub (recommended)
DATA_URL = 'https://raw.githubusercontent.com/ESpoorthy/NexTune/main/data/enhanced-headphones-dataset.csv'
df = pd.read_csv(DATA_URL)

# Option B: Load from Google Drive (uncomment if needed)
# df = pd.read_csv('/content/drive/MyDrive/NexTune/data/enhanced-headphones-dataset.csv')

print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

## 2. Data Cleaning

In [ ]:
df_clean = df.copy()

# Drop rows missing the target or brand
df_clean = df_clean.dropna(subset=['price_inr', 'brand'])
df_clean = df_clean[df_clean['price_inr'] > 0]  # remove zero-price entries

# Remove duplicates
df_clean = df_clean.drop_duplicates(subset=['product_name'], keep='first')

# Impute missing numerical values
df_clean['battery_life_hrs'] = df_clean.groupby('category')['battery_life_hrs'].transform(
    lambda x: x.fillna(x.median())
)
df_clean['bluetooth_version'] = df_clean['bluetooth_version'].fillna(
    df_clean['bluetooth_version'].mode()[0]
)
df_clean['driver_size_mm'] = df_clean.groupby('category')['driver_size_mm'].transform(
    lambda x: x.fillna(x.median())
)
df_clean['rating']                   = df_clean['rating'].fillna(df_clean['rating'].median())
df_clean['review_count']             = df_clean['review_count'].fillna(0)
df_clean['active_noise_cancellation']= df_clean['active_noise_cancellation'].fillna(0)
df_clean['mic_count']                = df_clean['mic_count'].fillna(2)

print(f'Clean dataset: {len(df_clean)} records')
print(f'Price range: ₹{df_clean.price_inr.min():.0f} – ₹{df_clean.price_inr.max():.0f}')

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Price distribution
axes[0].hist(df_clean['price_inr'], bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(df_clean['price_inr'].median(), color='red', linestyle='--',
                label=f'Median ₹{df_clean["price_inr"].median():.0f}')
axes[0].set_title('Price Distribution', fontweight='bold')
axes[0].set_xlabel('Price (INR)')
axes[0].legend()

# Price by category
df_clean.boxplot(column='price_inr', by='category', ax=axes[1])
axes[1].set_title('Price by Category', fontweight='bold')
axes[1].set_xlabel('')
plt.sca(axes[1])
plt.xticks(rotation=30, ha='right')
plt.suptitle('')

# Rating vs Price
axes[2].scatter(df_clean['rating'], df_clean['price_inr'], alpha=0.5, color='coral')
axes[2].set_title('Rating vs Price', fontweight='bold')
axes[2].set_xlabel('Rating')
axes[2].set_ylabel('Price (INR)')

plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
# Binary: has ANC
df_clean['has_anc'] = (df_clean['active_noise_cancellation'] == 1).astype(int)

# Value metric: price per hour of battery
df_clean['price_per_hour'] = df_clean['price_inr'] / (df_clean['battery_life_hrs'] + 1)

# Brand tier based on average price
brand_avg = df_clean.groupby('brand')['price_inr'].mean()
def brand_tier(brand):
    p = brand_avg.get(brand, 0)
    return 'premium' if p >= 10000 else ('mid' if p >= 3000 else 'budget')
df_clean['brand_tier'] = df_clean['brand'].apply(brand_tier)

# Major Bluetooth version
df_clean['bt_major'] = df_clean['bluetooth_version'].apply(
    lambda x: int(x) if pd.notna(x) else 5
)

# High rating flag
df_clean['high_rating'] = (df_clean['rating'] >= 4.0).astype(int)

# Water resistance flag
df_clean['has_ipx'] = df_clean['ipx_rating'].notna().astype(int)

print('✅ Feature engineering done')
print(df_clean[['has_anc','brand_tier','bt_major','high_rating','has_ipx']].head(3))

## 5. Encode & Prepare Feature Matrix

In [ ]:
label_encoders = {}
for col in ['category', 'brand', 'brand_tier']:
    le = LabelEncoder()
    df_clean[f'{col}_enc'] = le.fit_transform(df_clean[col].astype(str))
    label_encoders[col] = le

FEATURES = [
    'category_enc', 'brand_enc', 'brand_tier_enc',
    'rating', 'review_count',
    'battery_life_hrs', 'driver_size_mm',
    'bluetooth_version', 'bt_major',
    'mic_count', 'has_anc', 'has_ipx',
    'high_rating', 'price_per_hour'
]
TARGET = 'price_inr'

X = df_clean[FEATURES].fillna(0)
y = df_clean[TARGET]

print(f'Feature matrix: {X.shape}  |  Target: {y.shape}')

## 6. Train / Test Split & Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {len(X_train)}  |  Test: {len(X_test)}')

## 7. Train Multiple Models

In [ ]:
models = {
    'Linear Regression':      LinearRegression(),
    'Ridge':                  Ridge(alpha=1.0),
    'Lasso':                  Lasso(alpha=1.0),
    'Random Forest':          RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting':      GradientBoostingRegressor(n_estimators=100, random_state=42),
}

results = []

for name, model in models.items():
    model.fit(X_train_sc, y_train)
    y_pred = model.predict(X_test_sc)

    r2   = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / (y_test + 1))) * 100

    # 5-fold CV R²
    cv_r2 = cross_val_score(model, X_train_sc, y_train, cv=5, scoring='r2').mean()

    results.append({'Model': name, 'R²': r2, 'CV R²': cv_r2,
                    'RMSE': rmse, 'MAE': mae, 'MAPE%': mape})
    print(f'{name:25s}  R²={r2:.3f}  CV R²={cv_r2:.3f}  RMSE=₹{rmse:.0f}  MAE=₹{mae:.0f}')

results_df = pd.DataFrame(results).sort_values('R²', ascending=False)
print('\n', results_df.to_string(index=False))

## 8. Model Comparison Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# R² comparison
colors = ['#2ecc71' if r >= 0.70 else '#e74c3c' for r in results_df['R²']]
axes[0].barh(results_df['Model'], results_df['R²'], color=colors)
axes[0].axvline(0.70, color='black', linestyle='--', label='Target R²=0.70')
axes[0].set_title('R² Score by Model', fontweight='bold')
axes[0].set_xlabel('R²')
axes[0].legend()

# RMSE comparison
axes[1].barh(results_df['Model'], results_df['RMSE'], color='steelblue')
axes[1].set_title('RMSE by Model (lower = better)', fontweight='bold')
axes[1].set_xlabel('RMSE (₹)')

plt.tight_layout()
plt.show()

## 9. Best Model — Actual vs Predicted

In [ ]:
best_name  = results_df.iloc[0]['Model']
best_model = models[best_name]
y_pred_best = best_model.predict(X_test_sc)

print(f'Best model: {best_name}')
print(f'  R²   = {results_df.iloc[0]["R²"]:.4f}')
print(f'  RMSE = ₹{results_df.iloc[0]["RMSE"]:.2f}')
print(f'  MAE  = ₹{results_df.iloc[0]["MAE"]:.2f}')

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_best, alpha=0.6, color='teal', edgecolors='white', s=60)
lims = [min(y_test.min(), y_pred_best.min()), max(y_test.max(), y_pred_best.max())]
plt.plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
plt.xlabel('Actual Price (₹)', fontsize=12)
plt.ylabel('Predicted Price (₹)', fontsize=12)
plt.title(f'{best_name} — Actual vs Predicted', fontsize=14, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

## 10. Feature Importance

In [ ]:
# Works for tree-based models; falls back to coefficients for linear models
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    label = 'Feature Importance'
else:
    importances = np.abs(best_model.coef_)
    label = '|Coefficient|'

feat_imp = pd.Series(importances, index=FEATURES).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
feat_imp.plot(kind='barh', color='steelblue')
plt.title(f'Feature Importance — {best_name}', fontsize=14, fontweight='bold')
plt.xlabel(label)
plt.tight_layout()
plt.show()

print('\nTop 5 most important features:')
print(feat_imp.sort_values(ascending=False).head(5).to_string())

## 11. Save Model & Artifacts

In [ ]:
import os

# Save to Google Drive
SAVE_DIR = '/content/drive/MyDrive/NexTune/models'
os.makedirs(SAVE_DIR, exist_ok=True)

joblib.dump(best_model,      f'{SAVE_DIR}/price_predictor.pkl')
joblib.dump(scaler,          f'{SAVE_DIR}/scaler.pkl')
joblib.dump(label_encoders,  f'{SAVE_DIR}/label_encoders.pkl')

print(f'✅ Model saved  → {SAVE_DIR}/price_predictor.pkl')
print(f'✅ Scaler saved → {SAVE_DIR}/scaler.pkl')
print(f'✅ Encoders saved → {SAVE_DIR}/label_encoders.pkl')

## 12. Predict Price for a New Headphone

Simulate recommending a price for a new product your company is about to launch.

In [ ]:
# ✏️ Fill in your new headphone's specs here
new_headphone = {
    'category':        'true wireless earbuds',   # over-ear headphone / neckband / true wireless earbuds
    'brand':           'Boat',
    'brand_tier':      'budget',                  # budget / mid / premium
    'rating':          4.2,
    'review_count':    0,                         # new product — no reviews yet
    'battery_life_hrs':40,
    'driver_size_mm':  10,
    'bluetooth_version':5.3,
    'bt_major':        5,
    'mic_count':       4,
    'has_anc':         0,
    'has_ipx':         1,
    'high_rating':     1,
    'price_per_hour':  0,                         # unknown for new product
}

# Encode categorical fields
for col in ['category', 'brand', 'brand_tier']:
    le = label_encoders[col]
    val = new_headphone[col]
    new_headphone[f'{col}_enc'] = le.transform([val])[0] if val in le.classes_ else 0

# Build feature vector in the correct order
input_vec = np.array([[new_headphone.get(f, 0) for f in FEATURES]])
input_scaled = scaler.transform(input_vec)

predicted_price = best_model.predict(input_scaled)[0]

print('=' * 50)
print('💡 PRICE RECOMMENDATION')
print('=' * 50)
print(f'  Model used   : {best_name}')
print(f'  Predicted    : ₹{predicted_price:,.0f}')
print(f'  Suggested    : ₹{predicted_price * 0.95:,.0f} – ₹{predicted_price * 1.05:,.0f}  (±5% band)')
print('=' * 50)